Train Unet with Data Augmentation


In [2]:
import os
import glob

images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))

print("Number of images:", len(images_list))
print("Number of masks:", len(masks_list))
# برای دیدن اولین فایل‌ها
print("Sample images paths:", images_list[:5])
print("Sample masks paths:", masks_list[:5])


In [4]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image

# ======================
# 1) تابع محاسبه DICE
# ======================
def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)      # تبدیل لاجیت خروجی به محدوده [0,1]
    pred = (pred > 0.5).float()     # باینری کردن خروجی
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.mean()

# ======================
# 2) تنظیم seed
# ======================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ======================
# 3) تعریف دیتاست
# ======================
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")  # اگر تصاویر رنگی‌اند
        mask = Image.open(mask_path).convert("L")    # ماسک معمولاً خاکستری یا باینری

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        # باینری کردن ماسک (اگر داده‌های شما 0 و 255 هستند)
        mask = (mask > 0.5).float()

        return image, mask

# ======================
# 4) آدرس فولدر تصاویر
# ======================
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))
assert len(images_list) == len(masks_list), "تعداد تصاویر با ماسک‌ها برابر نیست."

# ======================
# 5) تقسیم داده‌ها: 80% Train, 20% Test
# ======================
total_size = len(images_list)
train_size = int(total_size * 0.8)
test_size  = total_size - train_size

train_images = images_list[:train_size]
train_masks  = masks_list[:train_size]
test_images  = images_list[train_size:]
test_masks   = masks_list[train_size:]

# ======================
# 6) تعریف ترنسفورم‌ها
# ======================
# الف) ترنسفورم پایه (بدون Data Augmentation) برای داده‌های تست
transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ب) ترنسفورم پایه برای داده‌های ترین
transform_train_base = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# پ) ترنسفورم آگومنت‌شده برای داده‌های ترین
# می‌توانید بسته به نیاز، افکت‌های دیگری هم اضافه کنید (چرخش‌های بیشتر، تغییر رنگ، ...)
transform_train_aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor()
])

# ======================
# 7) ساخت دیتاست
# ======================
# دیتاست اصلی
train_dataset_base = CorneaDataset(train_images, train_masks, transform=transform_train_base)
# دیتاست آگومنت‌شده
train_dataset_aug  = CorneaDataset(train_images, train_masks, transform=transform_train_aug)
# ترکیب دیتاست اصلی و آگومنت شده برای دوبرابر شدن تعداد نمونه‌های ترین
train_dataset      = ConcatDataset([train_dataset_base, train_dataset_aug])

test_dataset       = CorneaDataset(test_images, test_masks, transform=transform_test)

# ======================
# 8) تعریف دیتالودر
# ======================
batch_size = 2
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 9) تعریف مدل U-Net (ساده)
# ======================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        self.conv_down1 = DoubleConv(in_channels, 64)
        self.conv_down2 = DoubleConv(64, 128)
        self.conv_down3 = DoubleConv(128, 256)
        self.conv_down4 = DoubleConv(256, 512)
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = DoubleConv(512, 1024)
        
        self.uptrans1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)
        self.uptrans2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)
        self.uptrans3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)
        self.uptrans4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        self.output = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.conv_down1(x)
        x2 = self.maxpool(x1)

        x2 = self.conv_down2(x2)
        x3 = self.maxpool(x2)

        x3 = self.conv_down3(x3)
        x4 = self.maxpool(x3)

        x4 = self.conv_down4(x4)
        x5 = self.maxpool(x4)

        # Bottleneck
        x5 = self.bottleneck(x5)

        # Decoder
        x6 = self.uptrans1(x5)
        x6 = torch.cat([x4, x6], dim=1)
        x6 = self.conv_up1(x6)

        x7 = self.uptrans2(x6)
        x7 = torch.cat([x3, x7], dim=1)
        x7 = self.conv_up2(x7)

        x8 = self.uptrans3(x7)
        x8 = torch.cat([x2, x8], dim=1)
        x8 = self.conv_up3(x8)

        x9 = self.uptrans4(x8)
        x9 = torch.cat([x1, x9], dim=1)
        x9 = self.conv_up4(x9)

        out = self.output(x9)
        return out

# ======================
# 10) آماده سازی برای آموزش
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 25

# ======================
# 11) حلقه اصلی آموزش
# ======================
for epoch in range(num_epochs):
    print(f"\n===== [Epoch {epoch+1}/{num_epochs}] =====")
    
    # --- Training ---
    model.train()
    train_loss_epoch = 0.0
    train_dice_epoch = 0.0
    train_batches = 0

    for batch_idx, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks.float())  # توجه کنید ماسک float باشد
        loss.backward()
        optimizer.step()

        dice_val = dice_coefficient(outputs, masks).item()

        train_loss_epoch += loss.item()
        train_dice_epoch += dice_val
        train_batches += 1

        # پرینت در هر Batch
        print(f"[Train] Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f} | Dice: {dice_val:.4f}")

    avg_train_loss = train_loss_epoch / train_batches
    avg_train_dice = train_dice_epoch / train_batches
    
    # --- Testing ---
    model.eval()
    test_loss_epoch = 0.0
    test_dice_epoch = 0.0
    test_batches = 0

    with torch.no_grad():
        for batch_idx, (imgs, masks) in enumerate(test_loader):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks.float())
            dice_val = dice_coefficient(outputs, masks).item()

            test_loss_epoch += loss.item()
            test_dice_epoch += dice_val
            test_batches += 1

            # پرینت در هر Batch از تست
            print(f"[Test ] Epoch {epoch+1} | Batch {batch_idx+1}/{len(test_loader)}  | Loss: {loss.item():.4f} | Dice: {dice_val:.4f}")

    avg_test_loss = test_loss_epoch / test_batches
    avg_test_dice = test_dice_epoch / test_batches

    # پرینت خلاصه نهایی هر Epoch
    print(f"==> Epoch {epoch+1} Summary:")
    print(f"    Train - Avg Loss: {avg_train_loss:.4f}, Avg Dice: {avg_train_dice:.4f}")
    print(f"    Test  - Avg Loss: {avg_test_loss:.4f}, Avg Dice: {avg_test_dice:.4f}")

# ======================
# 12) ذخیره مدل
# ======================
model_path = "unet_cornea_segmentation_no_earlystop.pth"
torch.save(model.state_dict(), model_path)
print(f"\nمدل ذخیره شد در: {model_path}")
